# RAG Ingestion Layer
### Ein geführter Workshop zur ersten und kritischsten Stufe jedes RAG-Systems

---

## Bevor wir beginnen: Das große Bild

Bevor eine einzige Zeile Code ausgeführt wird, lohnt es sich, das System mental aufzubauen, denn wer die Ingestion Layer nicht versteht, begreift auch nicht, warum bestimmte
Entscheidungen so getroffen werden, wie sie getroffen werden.

---

### Was ist ein RAG-System?

RAG steht für **Retrieval-Augmented Generation**. Die Grundidee lautet:

> Anstatt ein Sprachmodell mit allem Wissen zu trainieren, das es jemals brauchen könnte,
> lassen wir es zur Laufzeit in einer Wissensbasis **suchen** und ihm die
> gefundenen Informationen als Kontext geben.

Ein RAG-System besteht aus zwei großen Teilen:

1. **Offline-Teil (Ingestion + Embedding + Indexing):**
   Rohdaten werden aufbereitet, in Vektoren umgewandelt und in einem Index gespeichert.
2. **Online-Teil (Retrieval + Generation):**
   Eine Query wird gestellt, relevante Chunks werden gefunden,
   das LLM antwortet mit diesem Kontext.

```
╔══════════════════════════════════════════════════════════════════╗
║  OFFLINE  (einmalig, oder bei Datenänderung)                    ║
║                                                                  ║
║  Rohdaten  →  Cleaning  →  Chunking  →  Persistenz              ║
║                                              │                   ║
║                                              ▼                   ║
║                                         Embedding               ║
║                                              │                   ║
║                                              ▼                   ║
║                                         Vektorindex             ║
╠══════════════════════════════════════════════════════════════════╣
║  ONLINE  (bei jeder Query)                                       ║
║                                                                  ║
║  Query  →  Embedding  →  Vektorsuche  →  Chunks  →  LLM         ║
╚══════════════════════════════════════════════════════════════════╝
```

---

### Was ist die Ingestion Layer, und warum ist sie kritisch?

Die Ingestion Layer ist **Stufe 1** im Offline-Teil. Sie ist die einzige Stelle
im gesamten System, an der Rohdaten berührt werden. Alles danach, also Embedding, Retrieval und die LLM-Antwort, setzt auf dem auf, was hier erzeugt wurde.

Das klingt nach Vorverarbeitung. Es ist keine.

**Retrieval-Qualität entscheidet sich hier.** Nicht im Retrieval-Modul.
Nicht im LLM. Hier.

Warum?

- Ein Embedding-Modell wandelt einen Textabschnitt in einen Vektor um.
  Dieser Vektor repräsentiert die **semantische Bedeutung** des Textes.
- Wenn der Text kein klares semantisches Zentrum hat, weil er mitten in einem Gedanken
  anfängt oder endet, zu groß zum Präzisieren oder zu klein für Kontext ist,
  dann ist der Vektor ebenfalls unscharf.
- Bei der Vektorsuche wird dieser unscharfe Vektor mit einer Query verglichen.
  Das Ergebnis: schlechte Treffer.
- Das LLM bekommt irrelevanten oder fragmentierten Kontext. Die Antwort wird entsprechend.

**Schlechte Chunks sind im nachgelagerten System nicht reparierbar.**
Kein Retrieval-Algorithmus, kein Reranker, kein Prompt rettet einen Chunk,
der semantisch inkohärent ist.

---

### Was passiert in diesem Notebook?

Wir durchlaufen die vollständige Ingestion-Pipeline:

```
Rohdaten  →  Cleaning  →  Chunking  →  Persistenz  →  für das Embedding vorbereitete Chunks
```

Wir werden dabei jede Designentscheidung begründen, jeden Output interpretieren,
und die Stellen benennen, an denen Dinge schiefgehen können.

Was wir **nicht** tun: Embedding, Vektordatenbanken, Retrieval, LLM-Inferenz.
Die werden nur so weit erwähnt, dass klar ist, wo Ingestion endet und was danach kommt.

---

## 1. Rohdaten: Was liegt vor?

Der erste Schritt ist immer derselbe: verstehen, womit wir es zu tun haben.

Ein häufiger Fehler ist, Ingestion-Code zu schreiben, ohne die Eingabe wirklich zu kennen.
Wie viele Dateien? Welche Formate? Wie groß? Wie strukturiert?

Diese Fragen sind keine Formalität. Sie bestimmen, welche Loader und Chunker sinnvoll sind und an welchen Stellen Fallbacks gebraucht werden.

In [ ]:
from pathlib import Path
from collections import Counter

raw_path = Path("/home/jovyan/shared/data/raw")

supported_extensions = {".md", ".jsonl"}
all_files = sorted(raw_path.rglob("*")) if raw_path.exists() else []
files = [f for f in all_files if f.is_file() and f.suffix in supported_extensions]

print(f"Verzeichnis  : {raw_path}")
print(f"Dateien      : {len(files)}")
print()

for f in files:
    print(f"  {f.suffix}  {f.name:<40}  {f.stat().st_size:>8} Bytes")

print()
for ext, count in Counter(f.suffix for f in files).most_common():
    print(f"  {ext}  →  {count} Datei(en)")

**Was dieser Output bedeutet:**

Wir sehen die Dateien, die als Rohdaten vorliegen. Jede Datei wird durch einen
entsprechenden **Loader** eingelesen, etwa der `.md`-Loader für Markdown
und der `.jsonl`-Loader für line-delimited JSON.

Loader tun mehr als `file.read_text()`. Sie normalisieren die Dateistruktur in ein
einheitliches Dokument-Format, das die Pipeline erwartet. Ein `.jsonl`-Loader zum
Beispiel iteriert über Zeilen, parst jede als JSON-Objekt und extrahiert das relevante
Textfeld, damit der Rest der Pipeline nicht wissen muss, wie die Rohdaten aussahen.

In [ ]:
md_files    = [f for f in files if f.suffix == ".md"]
jsonl_files = [f for f in files if f.suffix == ".jsonl"]

sample_md    = md_files[0]    if md_files    else None
sample_jsonl = jsonl_files[0] if jsonl_files else None

print(f"Markdown-Beispiel : {sample_md}")
print(f"JSONL-Beispiel    : {sample_jsonl}")

---

## 2. Rohdokument ansehen: Was ist der Ausgangszustand?

Bevor wir Cleaning und Chunking verstehen, müssen wir sehen, was **vor** diesen
Schritten vorhanden ist.

Rohdaten sind selten so sauber, wie wir es uns wünschen. In der Praxis enthalten sie:

- **Unsichtbare Steuerzeichen** (`\r`, `\r\n`, BOM `\ufeff`), je nach Betriebssystem und Quelle
- **Trailing Whitespace** am Zeilenende, oft aus Copy-Paste oder Editoren
- **Inkonsistente Leerzeilen**, manchmal drei Leerzeilen, manchmal keine
- **Fehlende Strukturierung**: ein 50-KB-Markdown-Dokument ist ein einzelnes, unteiltes Objekt

Das Problem: Kein Embedding-Modell der Welt kann mit einem 50-KB-Text sinnvoll umgehen.
Kontextlimits werden gesprengt. Vektoren werden unpräzise. Deshalb müssen Dokumente
geteilt werden. Darauf gehen wir beim Chunking ein.

In [ ]:
if sample_md is None:
    print("Keine .md Datei gefunden.")
else:
    raw_md = sample_md.read_text(encoding="utf-8")
    print(f"Datei   : {sample_md.name}")
    print(f"Groesse : {len(raw_md)} Zeichen")
    print()
    print("--- Erste 600 Zeichen (als repr – zeigt unsichtbare Zeichen) ---")
    print(repr(raw_md[:600]))

**Was wir im `repr`-Output suchen:**

- `\r\n` statt nur `\n`: Windows-Zeilenenden. Wenn nicht normalisiert, entstehen
  beim späteren Splitting inkonsistente Ergebnisse.
- `\ufeff` am Anfang: UTF-8 BOM. Viele Parser stolpern darüber oder behandeln es als Inhalt.
- `   ` (Leerzeichen am Zeilenende): Schleichen sich in Chunks und beeinflussen Zeichenzahlen.

Diese Zeichen sind mit dem bloßen Auge nicht sichtbar. `repr()` macht sie sichtbar,
deshalb verwenden wir es hier.

In [ ]:
import json

if sample_jsonl is None:
    print("Keine .jsonl Datei gefunden.")
else:
    with open(sample_jsonl, "r", encoding="utf-8") as f:
        first_line = next((l for l in f if l.strip()), None)

    print(f"Datei : {sample_jsonl.name}")
    if first_line:
        try:
            obj = json.loads(first_line)
            print(f"Felder : {list(obj.keys())}")
        except Exception:
            pass
        print(f"Raw    : {repr(first_line[:200])}")

**JSONL ist kein Dokument, sondern ein Stream.**

Eine `.jsonl`-Datei enthält pro Zeile ein eigenständiges JSON-Objekt. Das bedeutet:
Eine einzige `.jsonl`-Datei kann hunderte logisch getrennte Dokumente enthalten.
Der `JsonlLoader` iteriert über diese Zeilen und behandelt jede als eigenständiges Dokument.

Das erklärt auch, warum die Anzahl der Chunks später deutlich höher sein kann als
die Anzahl der Dateien: mehrere Dateien, viele Dokumente, viele Chunks.

---

## 3. Cleaning: Deterministisch normalisieren

### Das Problem, das Cleaning löst

Stellen wir uns vor, wir verarbeiten dasselbe Dokument zweimal: einmal auf einem
Linux-System, einmal auf Windows. Auf Windows hat das Dokument `\r\n` als Zeilenenden.
Wenn wir daraus einen SHA-256-Hash berechnen, erhalten wir **unterschiedliche Hashes**,
obwohl der Inhalt identisch ist.

Das ist das Problem. Eine `document_id` auf Basis eines Hashes ist nur dann zuverlässig,
wenn der Text, aus dem der Hash berechnet wird, **deterministisch** ist,
also immer identisch, egal wann und wo.

### Was Cleaning tut und was es nicht tut

Der `DefaultCleaner` **verändert den Inhalt nicht**. Er normalisiert nur die **Form**:

1. UTF-8 BOM entfernen (`\ufeff`)
2. Zeilenenden vereinheitlichen (`\r\n`, `\r` → `\n`)
3. Trailing Whitespace pro Zeile entfernen
4. Abschließende Leerzeilen entfernen
5. `None` zurückgeben, wenn nach Cleaning nichts übrig bleibt

Er macht kein Stemming, keine Stopwort-Entfernung, keine semantische Transformation.
Das würde Inhalte verändern, was hier nicht erwünscht ist.

**Reihenfolge ist hier nicht beliebig.** BOM muss als erstes entfernt werden,
da sonst die Zeichenende-Normalisierung auf einem Text arbeitet, der noch ein
führendes Sonderzeichen enthält.

In [ ]:
from rag.ingestion.cleaner import DefaultCleaner

cleaner = DefaultCleaner()

if sample_md is None:
    raise FileNotFoundError("Keine .md Datei fuer Cleaning-Demo gefunden.")

raw = sample_md.read_text(encoding="utf-8")
cleaned = cleaner.clean(raw)

print("VORHER (repr, erste 200 Zeichen):")
print(repr(raw[:200]))
print()
print("NACHHER (repr, erste 200 Zeichen):")
print(repr(cleaned[:200]) if cleaned else "None – Dokument war nach Cleaning leer")

**Was wir im Vorher/Nachher-Vergleich sehen sollten:**

- `\r\n` sollte zu `\n` werden
- Abschließende Leerzeichen am Zeilenende (`  \n`) sollten zu `\n` werden
- `\ufeff` am Anfang sollte verschwinden

Wenn die beiden `repr`-Ausgaben identisch aussehen: Das ist auch ein valides Ergebnis,
es bedeutet, dass das Dokument bereits sauber war. Cleaning ist dann eine Null-Operation,
aber eine *deterministische* Null-Operation.

In [ ]:
if cleaned is None:
    raise ValueError("Cleaner hat None zurueckgegeben – Dokument nach Cleaning leer.")

import hashlib

print(f"Laenge RAW        : {len(raw):>8} Zeichen")
print(f"Laenge CLEANED    : {len(cleaned):>8} Zeichen")
print(f"Entfernt          : {len(raw) - len(cleaned):>8} Zeichen")
print()

# Die document_id wird im System exakt so abgeleitet:
doc_id = "doc_" + hashlib.sha256(cleaned.encode("utf-8")).hexdigest()[:16]
print(f"document_id : {doc_id}")
print()
print("Warum das funktioniert:")
print("  Gleicher Input → gleicher SHA-256 → gleiche ID → stabiler Vektorindex.")
print("  Ohne Determinismus: gleiche Datei, zwei Laeufe, zwei IDs → Duplikate im Index.")

### Warum Determinismus systemkritisch ist

Stellen wir uns vor, Cleaning wäre nicht deterministisch. Vielleicht normalisiert es
manchmal Leerzeichen, manchmal nicht, abhängig von internem State oder Seiteneffekten.

Was passiert:

```
Erster Ingestion-Lauf:  document_id = "doc_a1b2c3d4..."
Zweiter Lauf (gleiche Datei, nicht-deterministisches Cleaning):
                        document_id = "doc_x7y8z9w0..."
```

Der Vektorindex enthält jetzt **zwei Einträge für dasselbe Dokument**, mit
unterschiedlichen IDs. Deduplication schlägt fehl. Index-Updates können nicht mehr
gezielt angewendet werden. Reproduzierbarkeit ist nicht mehr gegeben.

Deterministisches Cleaning ist also keine ästhetische Präferenz,
sondern eine Architektur-Anforderung.

---

## 4. Chunking: Das schwierigste Problem der Ingestion

### Warum Chunking überhaupt?

Embedding-Modelle können keine beliebig langen Texte verarbeiten. Ein typisches Modell
hat ein Limit von 512 bis 8192 Tokens. Ein größeres Dokument wie ein technisches Manual,
ein Gesetzestext oder ein Wiki-Artikel überschreitet dieses Limit bei Weitem.

Aber das Token-Limit ist nicht der einzige Grund. Selbst wenn ein Modell ein ganzes
Dokument verarbeiten könnte, wäre der resultierende Vektor **semantisch unscharf**.
Ein Vektor für ein 20-seitiges Dokument hat keinen klaren Schwerpunkt.
Er repräsentiert alles und deshalb nichts Bestimmtes.

Bei Retrieval wird dieser Vektor mit einer konkreten Query verglichen.
Eine Query ist präzise. Ein Gesamt-Dokument-Vektor ist unscharf. Das Match ist schlecht.

---

### Was ist ein guter Chunk?

Ein guter Chunk erfüllt folgende Eigenschaften:

| Eigenschaft | Warum sie wichtig ist |
|-------------|----------------------|
| **Semantisch kohärent** | Der Text behandelt ein Thema oder einen Gedanken. Der Vektor hat einen klaren Schwerpunkt. |
| **Vollständig** | Kein abgeschnittener Satz, kein fehlender Kontext. Das LLM kann den Chunk verstehen, ohne das Original zu sehen. |
| **Weder zu groß noch zu klein** | Zu groß: unscharf. Zu klein: kein Kontext, kaum Signal. |
| **An logischen Grenzen endend** | Absatz-, Abschnitts-, Satzgrenzen, also dort, wo das Dokument selbst eine Grenze signalisiert. |

---

### Was ist ein schlechter Chunk? (Mit Beispiel)

Stellen wir uns diesen Text vor:

```
Die Transformerarchitektur basiert auf dem Attention-Mechanismus, der es dem Modell
ermöglicht, relevante Teile des Inputs zu gewichten. Im Gegensatz zu RNNs verarbeitet
ein Transformer alle Token gleichzeitig. Das macht Training deutlich effizienter.

Ein wichtiger Hyperparameter ist die Anzahl der Attention-Heads.
```

Ein schlechter Chunk durch blindes Zeichenschneiden bei Position 150:

```
Chunk 1: "Die Transformerarchitektur basiert auf dem Attention-Mechanismus, der es dem"
Chunk 2: "Modell ermöglicht, relevante Teile des Inputs zu gewichten. Im Gegensatz zu"
```

Chunk 1 endet mitten im Satz. Chunk 2 beginnt mit einem Satzfragment ohne Kontext.

Ein guter Chunk würde den ersten Absatz als Einheit behandeln:

```
Chunk 1: "Die Transformerarchitektur basiert auf dem Attention-Mechanismus, der es dem
Modell ermöglicht, relevante Teile des Inputs zu gewichten. Im Gegensatz zu RNNs
verarbeitet ein Transformer alle Token gleichzeitig. Das macht Training deutlich effizienter."

Chunk 2: "Ein wichtiger Hyperparameter ist die Anzahl der Attention-Heads."
```

---

### Warum Overlap existiert

Selbst bei guten Chunk-Grenzen gibt es ein Problem: Kontext an den Rändern.
Der letzte Satz eines Chunks und der erste Satz des nächsten Chunks sind semantisch
verwandt, sie gehören zum selben Gedankenfluss.

**Overlap** löst das, indem der letzte Teil eines Chunks mit dem Anfang des
nächsten wiederholt wird:

```
Text:    [----Chunk 0----][----Chunk 1----][----Chunk 2----]

Mit Overlap:
Chunk 0: [==========]
Chunk 1:         [==========]
Chunk 2:                 [==========]
                 ↑↑↑↑↑↑↑ Overlap-Bereich
```

Der Preis: mehr Chunks, mehr Speicher, mehr Embedding-Aufwand.
Der Gewinn: weniger fehlende Kontext-Information an Chunk-Grenzen.

---

## 5. Chunking-Strategien im Überblick

Es gibt viele Wege, einen Text zu teilen. Jeder hat Stärken, Schwächen und einen Anwendungskontext:

| Strategie | Prinzip | Stärke | Schwäche |
|-----------|---------|--------|----------|
| **Fixed Character Window** | Schneidet bei fixer Zeichenzahl | Simpel, robust | Blind für Satz- und Absatzgrenzen |
| **Sliding Window** | Fixed Window + Overlap | Kein Kontextverlust an Grenzen | Immer noch semantisch blind |
| **Sentence-based** | Schneidet an Satzgrenzen | Sätze bleiben intakt | Erfordert Sentence-Tokenizer, teurer |
| **Structure-aware** | Nutzt Dokumentstruktur (z.B. Markdown-AST) | Semantisch kohärent, natürliche Grenzen | Erfordert parsbares Dokument |
| **Semantic chunking** | Misst semantische Distanz zwischen Sätzen | Präziseste semantische Grenzen | Sehr rechenintensiv, erfordert Embeddings |
| **Token-based** | Schneidet nach Token-Anzahl | Exakt für Modell-Limits | Abhängig vom Tokenizer |

In diesem Notebook verwenden wir **SlidingWindowChunker** und **DocumentStructureChunker**,
einen robusten Fallback und eine strukturbewusste Strategie.

---

## 6. Strategie 1: SlidingWindowChunker

Der SlidingWindowChunker ist die grundlegendste Strategie: Er schneidet Text in Fenster
fixer Zeichenzahl, mit einem konfigurierbaren Overlap.

**Was er macht:**
- Nimmt den Text als flachen String
- Erzeugt Chunk 0: Zeichen 0 bis `chunk_size`
- Erzeugt Chunk 1: Zeichen `chunk_size - overlap` bis `2 * chunk_size - overlap`
- Und so weiter

**Was er nicht macht:**
- Er weiß nicht, was ein Satz ist
- Er weiß nicht, was eine Überschrift ist
- Er weiß nicht, wo ein Gedanke anfängt und wo er aufhört

**Wann er trotzdem sinnvoll ist:**
- Fallback für nicht-strukturierten Text
- Plain Text ohne Markdown oder HTML
- Situationen, in denen kein Parser verfügbar oder zuverlässig ist

In [ ]:
from rag.ingestion.chunking.sliding_window_chunker import SlidingWindowChunker
from rag.ingestion.chunking.strategies import FIXED_OVERLAP

if sample_md is None:
    raise FileNotFoundError("Keine .md Datei fuer Chunking-Demo.")

md_text = cleaner.clean(sample_md.read_text(encoding="utf-8"))
if md_text is None:
    raise ValueError("Markdown nach Cleaning leer.")

sliding_chunker = SlidingWindowChunker(
    chunk_size=300,
    overlap=50,
    strategy=FIXED_OVERLAP,
)

md_doc = {
    "id":       "demo_md",
    "content":  md_text,
    "metadata": {"source": str(sample_md), "type": "md"},
}

sliding_chunks = list(sliding_chunker.chunk(md_doc))

print(f"Konfiguration: chunk_size=300, overlap=50")
print(f"Chunks erzeugt: {len(sliding_chunks)}")
print()

for i, chunk in enumerate(sliding_chunks[:3]):
    print(f"--- Chunk {i} ({len(chunk['text'])} Zeichen) ---")
    print(f"Anfang : {repr(chunk['text'][:80])}")
    print(f"Ende   : {repr(chunk['text'][-80:])}")
    print()

**Was wir in diesem Output sehen sollten und was das bedeutet:**

Achte auf **Anfang** und **Ende** jedes Chunks. Fragen, die wir stellen sollten:

- Endet Chunk 0 mitten in einem Wort? Mitten in einem Satz?
  Das ist das SlidingWindow ohne Satzgrenzen-Bewusstsein.
- Beginnt Chunk 1 mit einem Wortfragment?
  Das ist die direkte Konsequenz des blinden Zeichenschnitts.
- Wie viel Overlap ist sichtbar zwischen Chunk 0 und Chunk 1?

**Die Abwägung:** Dieser Chunker ist deterministisch, schnell und unkompliziert zu prüfen.
Aber er erzeugt Chunks mit unklaren semantischen Grenzen, was direkt die Präzision
der späteren Embeddings beeinflusst.

Die entscheidende Frage: Wie groß ist der Schaden in der Praxis? Das hängt stark
vom Text ab. Fließtext ohne viele strukturelle Grenzen verliert wenig.
Dokumente mit klaren Abschnitten, Überschriften und Listen verlieren erheblich.

---

## 7. Strategie 2: DocumentStructureChunker

Dieser Chunker versteht die Struktur des Dokuments. Statt blind nach Zeichen zu schneiden,
parst er den Markdown-AST (Abstract Syntax Tree) und behandelt strukturelle Einheiten
als Chunk-Kandidaten:

- Überschriften und ihr folgender Inhalt werden zusammengehalten
- Absätze werden als Einheit behandelt
- Listen bleiben intakt
- Codeblöcke werden nicht mitten im Code getrennt

**Was passiert, wenn der Text nicht parsbar ist?**

Hier kommt der **Fallback-Mechanismus** ins Spiel. Wenn der Markdown-Parser keinen
AST extrahieren kann, weil der Text keine Markdown-Struktur hat oder weil er
Plain Text ist, greift automatisch der `SlidingWindowChunker` als Fallback.

Dieser Fallback ist keine Notlösung. Er ist eine bewusste Design-Entscheidung:
Strukturbewusste Strategie wo möglich, robuste Basis-Strategie wo nötig.

In [ ]:
from rag.ingestion.chunking.document_structure_chunker import DocumentStructureChunker
from rag.ingestion.chunking.strategies import STRUCTURE_AWARE

structure_chunker = DocumentStructureChunker(
    max_chunk_size=600,
    fallback_chunker=SlidingWindowChunker(
        chunk_size=300,
        overlap=50,
        strategy=FIXED_OVERLAP,
    ),
    strategy=STRUCTURE_AWARE,
)

structure_chunks = list(structure_chunker.chunk(md_doc))

print(f"Konfiguration: max_chunk_size=600, Fallback: SlidingWindowChunker")
print(f"Chunks erzeugt: {len(structure_chunks)}")
print()

for i, chunk in enumerate(structure_chunks[:3]):
    meta     = chunk.get("metadata", {})
    fallback = meta.get("fallback", False)
    strategy = meta.get("strategy", "?")
    print(f"--- Chunk {i} ({len(chunk['text'])} Zeichen | strategy={strategy} | fallback={fallback}) ---")
    print(chunk["text"][:280])
    print()

**Was wir hier genau analysieren sollten:**

**1. `fallback=True` in den Metadaten:**

Wenn ein Chunk `fallback=True` enthält, bedeutet das: Der Markdown-Parser konnte
diesen Abschnitt nicht sinnvoll strukturieren, er wurde durch den `SlidingWindowChunker`
verarbeitet. Das ist transparent und korrekt dokumentiert, aber es bedeutet auch, dass
dieser Chunk dieselben semantischen Grenzen-Probleme wie ein reiner SlidingWindow-Chunk
haben kann.

**2. Chunk-Grenzen direkt untersuchen:**

Schaue dir die ersten und letzten Zeilen der Chunks an. Beginnen sie mit einer Überschrift?
Enden sie mit einem abgeschlossenen Absatz? Das sind Zeichen, dass der Strukturchunker
korrekt arbeitet.

**3. Kritischer Hinweis zur Ehrlichkeit über Grenzen:**

Auch der `DocumentStructureChunker` erzeugt keine perfekten Chunks in jedem Fall.
Wenn ein einzelner Absatz länger als `max_chunk_size` ist, muss er trotzdem geteilt werden,
und das passiert durch den Fallback oder durch internes Splitting, das möglicherweise
nicht satzgenau ist. Die Strategie verbessert die Chunk-Qualität im Schnitt erheblich,
aber sie ist keine Garantie für semantische Vollständigkeit.

### Direkter Vergleich: SlidingWindow vs. DocumentStructure

Schauen wir uns an, wie sich Chunk-Anzahl und Größenverteilung unterscheiden:

In [ ]:
def chunk_stats(chunks, label):
    lengths = [len(c["text"]) for c in chunks]
    if not lengths:
        print(f"{label}: keine Chunks")
        return
    print(f"--- {label} ---")
    print(f"  Chunks        : {len(lengths)}")
    print(f"  Min (Zeichen) : {min(lengths)}")
    print(f"  Max (Zeichen) : {max(lengths)}")
    print(f"  Avg (Zeichen) : {round(sum(lengths)/len(lengths))}")
    print()

chunk_stats(sliding_chunks,   "SlidingWindowChunker (chunk_size=300, overlap=50)")
chunk_stats(structure_chunks, "DocumentStructureChunker (max_chunk_size=600)")

**Wie wir diese Zahlen interpretieren:**

- **Chunk-Anzahl:** Der `DocumentStructureChunker` erzeugt in der Regel weniger, aber
  größere Chunks, weil er Absätze als Einheit zusammenhält. Wenn die Chunk-Anzahl
  gleich oder ähnlich ist wie beim SlidingWindow, deutet das darauf hin, dass der
  Fallback häufig aktiviert wird. Der Strukturchunker hat dann wenig echte Dokumentstruktur
  vorgefunden.

- **Min/Max-Spanne:** SlidingWindow-Chunks haben eine enge Größenspanne (nahe `chunk_size`).
  StructureChunker-Chunks variieren stärker: kurze Abschnitte bleiben kurz,
  lange bleiben zusammen bis `max_chunk_size`.

- **Wenn Min sehr klein ist:** Ein sehr kleiner Chunk (wenige Zeichen) deutet auf einen
  isolierten Satz oder eine einzelne Überschrift ohne Inhalt hin.
  Solche Chunks sind wenig nützlich für Embedding.

---

## 8. Anatomie eines Chunks

Jeder Chunk ist nicht nur Text. Er trägt Metadaten, die ihn im System eindeutig
identifizierbar und später nachvollziehbar machen.

Schauen wir uns den genauen Aufbau an:

In [ ]:
if structure_chunks:
    c = structure_chunks[0]
    print("Vollständige Chunk-Struktur (erster Chunk):")
    print()
    print(f"  id          : {c['id']}")
    print(f"  document_id : {c['document_id']}")
    print(f"  text[:120]  : {repr(c['text'][:120])}")
    print(f"  metadata    :")
    for key, val in c["metadata"].items():
        print(f"    {key:<25}: {val}")

**Warum jedes Feld existiert:**

| Feld | Wozu es verwendet wird | Was passiert ohne es |
|------|------------------------|----------------------|
| `id` | Primärschlüssel im Vektorindex. Über diese ID wird der Vektor gespeichert und der Chunk-Text später wiedergefunden. | Kein eindeutiger Index-Eintrag möglich. |
| `document_id` | Verbindung zum Quelldokument. Ermöglicht: Alle Chunks eines Dokuments finden, Dokument aktualisieren und nur seine Chunks neu einbetten. | Chunks sind von ihrer Quelle getrennt. |
| `text` | Das, was eingebettet wird. Genau dieser String geht ins Embedding-Modell. | Kein Embedding möglich. |
| `metadata` | Nachvollziehbarkeit: Welches Dokument, welche Strategie, welche Position. | Retrieval-Ergebnisse sind nicht nachvollziehbar. |

**Besonders wichtig:** Die `id` des Chunks wird deterministisch aus `document_id` und
der Position im Dokument abgeleitet. Das bedeutet: Wenn ein Dokument unverändert
neu ingested wird, erhalten die Chunks dieselben IDs, und der Vektorindex kann
gezielt aktualisiert werden, ohne alle anderen Chunks zu berühren.

---

## 9. Pipeline konfigurieren: Alles zusammensetzen

Bisher haben wir die einzelnen Komponenten isoliert betrachtet. Jetzt verbinden wir
sie zu einer Pipeline.

Das Kompositionsmodell dieser Pipeline ist explizit: Jede Komponente wird namentlich
konfiguriert und an ihre Rolle gebunden. Das ist kein Zufall, denn es macht die Pipeline
gezielt diagnostizier- und testbar. Wenn ein Chunk unerwartete Eigenschaften hat, können wir genau
nachvollziehen, welcher Loader, Cleaner oder Chunker ihn erzeugt hat.

**Ein wichtiger Schritt hier:** Wir führen zunächst einen **Trockenlauf** durch
(`persist=False`). Damit verarbeiten wir alle Dokumente vollständig, speichern
aber nichts auf Disk. Das erlaubt uns, die Pipeline zu prüfen, ohne Seiteneffekte zu erzeugen.

In [ ]:
from rag.ingestion.loaders.md_loader import MdLoader
from rag.ingestion.loaders.jsonl_loader import JsonlLoader
from rag.ingestion.composition import IngestionComponents, LoaderBinding, IngestionRequest
from rag.ingestion.ingestion_api import create_ingestion_service, run_ingestion

primary_chunker = DocumentStructureChunker(
    max_chunk_size=600,
    fallback_chunker=SlidingWindowChunker(
        chunk_size=400,
        overlap=80,
        strategy=FIXED_OVERLAP,
    ),
    strategy=STRUCTURE_AWARE,
)

components = IngestionComponents(
    loader_bindings=(
        LoaderBinding(".md",    MdLoader()),
        LoaderBinding(".jsonl", JsonlLoader()),
    ),
    cleaner=cleaner,
    chunker=primary_chunker,
    doc_store=None,
    chunk_store=None,
)

print("Pipeline-Konfiguration:")
print()
for binding in components.loader_bindings:
    print(f"  {binding.extension}  →  {type(binding.loader).__name__}")
print(f"  Cleaner  : {type(components.cleaner).__name__}")
print(f"  Chunker  : {type(components.chunker).__name__}")
print(f"    Fallback: SlidingWindowChunker (chunk_size=400, overlap=80)")

In [ ]:
service     = create_ingestion_service(components)
request     = IngestionRequest(source=raw_path, persist=False)
dry_chunks  = list(run_ingestion(service, request))

print(f"Trockenlauf (persist=False) – vollständige Verarbeitung, keine Schreiboperation")
print(f"Chunks gesamt: {len(dry_chunks)}")
print()

from collections import defaultdict
strat_dist = defaultdict(int)
for c in dry_chunks:
    strat_dist[c["metadata"].get("strategy", "?")] += 1

print("Verteilung nach Strategie:")
for strat, count in sorted(strat_dist.items()):
    pct = round(100 * count / len(dry_chunks)) if dry_chunks else 0
    print(f"  {strat:<30} {count:>5} Chunks  ({pct}%)")

**Was uns die Strategie-Verteilung sagt:**

- **Hoher STRUCTURE_AWARE-Anteil:** Die Dokumente haben gut erkennbare Markdown-Struktur.
  Der Chunker konnte sinnvolle Grenzen setzen.
- **Hoher FIXED_OVERLAP-Anteil (Fallback):** Viele Dokumentabschnitte waren nicht
  strukturiert. In solchen Fällen griff der Fallback.

Wenn der Fallback-Anteil überraschend hoch ist: Das ist ein Signal, die Rohdaten zu
untersuchen. Haben die Markdown-Dateien wirklich eine klare Struktur?

**Warum `docs_loaded` > Dateianzahl sein kann:**

Eine `.jsonl`-Datei kann hunderte Zeilen enthalten, und jede Zeile ist ein eigenständiges
Dokument. Der `JsonlLoader` iteriert über alle Zeilen.

---

## 10. Persistenz: Die Grenze zwischen Ingestion und Embedding

### Warum Persistenz separat ist und warum das wichtig ist

Persistenz ist mehr als "Speichern". Sie ist eine **Architekturgrenze**.

Ohne diese Grenze müsste das Embedding-Modul Rohdaten laden, Cleaning durchführen,
Chunking durchführen, und das bei jedem Lauf erneut. Das hat mehrere Konsequenzen:

1. **Keine unabhängige Evaluierung:** Wenn Chunking und Embedding gekoppelt sind,
   kann man nicht dasselbe Chunkset mit zwei verschiedenen Embedding-Modellen vergleichen.
2. **Keine Reproduzierbarkeit:** Wenn der Ingestion-Prozess Seiteneffekte hat,
   variiert das Ergebnis zwischen Läufen.
3. **Keine saubere Testbarkeit:** Embedding-Modul, das auch chunked, ist schwerer
   zu testen als eines, das nur liest und embeddet.

Mit Persistenz:

```
Ingestion-Lauf  →  chunks.jsonl  (einmalig erzeugt, stabil)
Embedding-Lauf  →  liest chunks.jsonl  (ohne Rohdaten zu kennen)
```

Das Embedding-Modul muss nicht wissen, ob die Quelle Markdown oder JSONL war,
ob ein Fallback-Chunker aktiv war, oder wie die Originaldateien heißen.

### Deduplication

Die `InMemoryDedupStrategy` stellt sicher, dass dieselbe Chunk-ID nicht zweimal
in `chunks.jsonl` landet, auch wenn die Pipeline versehentlich zweimal über
dasselbe Dokument läuft. Das schützt vor aufgeblähten Stores und doppelten Embeddings.

In [ ]:
from rag.ingestion.storage.jsonl_store import JSONLStore
from rag.ingestion.storage.dedup import InMemoryDedupStrategy
from rag.ingestion.metrics import IngestionMetrics

processed_dir = Path("/home/jovyan/workspace/data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

doc_store_path   = processed_dir / "docs.jsonl"
chunk_store_path = processed_dir / "chunks.jsonl"

# Sauberer Start: Bestehende Dateien entfernen
for p in [doc_store_path, chunk_store_path]:
    if p.exists():
        p.unlink()
        print(f"Vorherige Datei entfernt: {p}")

print(f"Speicherort: {processed_dir}")
print()
print("Zieldateien:")
print(f"  docs.jsonl   – ein Eintrag pro Quelldokument (nach Cleaning)")
print(f"  chunks.jsonl – ein Eintrag pro Chunk (nach Chunking)")
print()
print("Das Embedding-Modul wird später ausschliesslich chunks.jsonl lesen.")

In [ ]:
doc_store   = JSONLStore(doc_store_path,   dedup=InMemoryDedupStrategy())
chunk_store = JSONLStore(chunk_store_path, dedup=InMemoryDedupStrategy())

components_with_store = IngestionComponents(
    loader_bindings=(
        LoaderBinding(".md",    MdLoader()),
        LoaderBinding(".jsonl", JsonlLoader()),
    ),
    cleaner=cleaner,
    chunker=primary_chunker,
    doc_store=doc_store,
    chunk_store=chunk_store,
)

service_with_store = create_ingestion_service(components_with_store)
persist_request    = IngestionRequest(source=raw_path, persist=True)
metrics            = IngestionMetrics()

persisted_chunks = list(run_ingestion(service_with_store, persist_request, metrics=metrics))

print(f"Persistenz-Lauf abgeschlossen.")
print(f"Chunks erzeugt und gespeichert: {len(persisted_chunks)}")
print()
print("Metriken:")
for key, val in metrics.as_dict().items():
    print(f"  {key:<35} {val}")

**Was die Metriken zeigen:**

Die Ingestion-Metriken geben Auskunft über den gesamten Lauf: wie viele Dokumente
geladen, wie viele bereinigt, wie viele verworfen wurden (Cleaning ergab `None`), wie viele
Chunks erzeugt und wie viele dedupliziert wurden.

Wenn `docs_dropped > 0`: Mindestens ein Dokument war nach dem Cleaning leer. Das kann
vorkommen, wenn eine Datei nur Whitespace enthält oder vollständig leer ist.

Wenn `chunks_deduped > 0`: Die Dedup-Strategie hat eingegriffen. In einem sauberen
ersten Lauf sollte das null sein.

---

## 11. Integritätsprüfung: Vertrauen, aber verifizieren

Nach dem Schreiben von Daten auf Disk gibt es keine Garantie, dass alles korrekt
angekommen ist. Dateisystem-Fehler, Encoding-Probleme, abgebrochene Schreiboperationen:
all das kann zu Datenverlust führen, der erst später auffällt.

Deshalb verifizieren wir nach dem Persistenz-Lauf **explizit und hart**: Wir laden
die Datei zurück und vergleichen sie mit dem, was wir geschrieben haben.

**Hard checks statt stillem Ignorieren:** Wenn eine Abweichung gefunden wird, wird ein
`ValueError` geworfen, nicht nur eine Warnung geloggt.

In [ ]:
from rag.ingestion.storage.chunk_loader import load_chunks
from rag.ingestion.storage.integrity import verify_chunk_counts, verify_chunk_ids

reloaded = list(load_chunks(chunk_store_path))

print(f"Chunks geschrieben : {len(persisted_chunks)}")
print(f"Chunks geladen     : {len(reloaded)}")
print()

# verify_chunk_counts: Wirft ValueError wenn Anzahl nicht uebereinstimmt.
verify_chunk_counts(
    expected=len(persisted_chunks),
    path=chunk_store_path,
)

# verify_chunk_ids: Wirft ValueError wenn eine ID fehlt oder eine unerwartete ID auftaucht.
verify_chunk_ids(
    expected_ids={c["id"] for c in persisted_chunks},
    path=chunk_store_path,
)

print("Integritaet OK: Anzahl und IDs stimmen ueberein.")

In [ ]:
# Längenverteilung der Chunk-Texte – als Qualitätsindikator
lengths = [len(c["text"]) for c in reloaded]

if lengths:
    print("Laengenverteilung der Chunk-Texte:")
    print()
    print(f"  Anzahl Chunks     : {len(lengths)}")
    print(f"  Min (Zeichen)     : {min(lengths)}")
    print(f"  Max (Zeichen)     : {max(lengths)}")
    print(f"  Avg (Zeichen)     : {round(sum(lengths)/len(lengths))}")
    print(f"  Gesamt (Zeichen)  : {sum(lengths)}")
    print()

    # Sehr kleine Chunks als potenzielles Qualitätsproblem hervorheben
    tiny_chunks = [c for c in reloaded if len(c["text"]) < 50]
    if tiny_chunks:
        print(f"  Hinweis: {len(tiny_chunks)} Chunk(s) unter 50 Zeichen – pruefen ob semantisch nuetzlich.")
        for c in tiny_chunks[:3]:
            print(f"    {repr(c['text'])}")

**Wie wir die Längenverteilung lesen:**

- **Durchschnitt nahe max_chunk_size:** Der Chunker füllt Chunks gut aus.
- **Sehr kleine Min-Werte (< 50 Zeichen):** Einzelne isolierte Elemente, z.B. eine
  Überschrift ohne folgenden Inhalt. Diese Chunks haben wenig semantischen Gehalt
  und produzieren schwache Embeddings.
- **Sehr große Max-Werte (weit über max_chunk_size):** Sollte eigentlich nicht vorkommen.
  Falls doch, deutet es auf ein Konfigurationsproblem hin.

**Zeichenanzahl ≠ semantische Qualität.** Ein 400-Zeichen-Chunk kann brillant oder
wertlos sein, abhängig davon, ob er einen kohärenten Gedanken enthält oder vier
zufällige Satzfragmente.

---

## 12. Embedding-Bereitschaft: Übergabe an die nächste Stufe

Die Ingestion ist abgeschlossen. Die letzte Aufgabe ist die formale Übergabe:
Wir prüfen, ob `chunks.jsonl` alle Voraussetzungen erfüllt, die das Embedding-Modul erwartet.

Das Embedding-Modul macht genau eine Sache:

```
chunk["text"]  →  Embedding-Modell  →  Vektor  →  Vektorindex
```

Dafür braucht es:
- `text`: nicht-leer, String
- `id`: eindeutig, vorhanden
- `document_id`: für spätere Verknüpfung
- `metadata`: mit `strategy`-Feld für Nachvollziehbarkeit

Was es **nicht** braucht: Rohdaten, Loader, Cleaner, Chunker.
Es weiß nicht mal, wie die ursprünglichen Dateien hießen.

In [ ]:
n   = len(reloaded)
ids = [c["id"] for c in reloaded]

checks = {
    "chunks.jsonl vorhanden"        : chunk_store_path.exists(),
    "Mindestens 1 Chunk"            : n > 0,
    "text nicht-leer"               : all(isinstance(c["text"], str) and c["text"] for c in reloaded),
    "id vorhanden"                  : all(c["id"] for c in reloaded),
    "IDs eindeutig"                 : len(set(ids)) == n,
    "document_id vorhanden"         : all(c["document_id"] for c in reloaded),
    "metadata vorhanden"            : all(isinstance(c["metadata"], dict) for c in reloaded),
    "metadata[strategy] vorhanden"  : all("strategy" in c["metadata"] for c in reloaded),
}

all_ok = all(checks.values())

print("Embedding-Readiness-Check:")
print()
for label, result in checks.items():
    status = "OK  " if result else "FAIL"
    print(f"  [{status}]  {label}")

print()
if not all_ok:
    raise RuntimeError("Embedding-Readiness-Check fehlgeschlagen – siehe FAIL oben.")

print(f"EMBEDDING-READY  ({n} Chunks in {chunk_store_path})")

---

## 13. Zusammenfassung: Was wir gelernt haben

---

### Cleaning: Determinismus als Architektur-Anforderung

Cleaning ist keine Vorverarbeitung. Es ist die Grundlage für reproduzierbare IDs,
stabile Indexe und zuverlässige Deduplication. Gleicher Input → gleicher SHA-256 →
gleiche `document_id`. Ohne Determinismus bricht das gesamte Update- und Dedup-Modell.

---

### Chunking: Wo Retrieval-Qualität entschieden wird

Ein Embedding-Modell ist nur so gut wie der Text, den es bekommt. Ein Vektor für einen
semantisch inkohärenten Chunk ist unscharf, und ein unscharfer Vektor führt zu schlechten
Retrieval-Ergebnissen. Kein Reranker und kein Prompt repariert das im Nachhinein.

Der `DocumentStructureChunker` verbessert die semantische Qualität erheblich, ist aber
kein Allheilmittel. Der Fallback-Mechanismus ist transparent: Chunks aus dem Fallback
tragen `strategy=FIXED_OVERLAP` in ihren Metadaten.

---

### Persistenz: Die Architekturgrenze

Ingestion erzeugt `chunks.jsonl`. Embedding liest `chunks.jsonl`. Dazwischen liegt
eine klare Grenze. Das ermöglicht:

- Verschiedene Embedding-Modelle auf denselben Chunks evaluieren
- Ingestion unabhängig von Embedding testen
- Reproduzierbarkeit: Gleiche Chunks, gleiche Vektoren, deterministischer Index

---

### Was als nächstes kommt

```
chunks.jsonl
    │
    ▼
  load_chunks(chunk_store_path)
    │
    ▼
  Embedding-Modell(chunk["text"])  →  Vektor
    │
    ▼
  Vektorindex.insert(chunk["id"], vektor, chunk["metadata"])
    │
    ▼
  Retrieval: Query  →  Vektorsuche  →  Top-k Chunks
    │
    ▼
  LLM bekommt Chunk-Texte als Kontext  →  Antwort
```

Die Ingestion ist abgeschlossen. Alles, was danach passiert, baut auf der Qualität auf,
die hier entschieden wurde.